# Multi-Timepoint PINN Training

Trains one PINN per patient using ALL their scans (2-6 timepoints), so the
network sees the full tumor trajectory and learns the dynamics. Then uses
learned parameters with a finite-volume solver for prediction.

**Setup:**
Data should be in Google Drive at `My Drive/STS_Project_Data/data/`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install
!rm -rf /content/BINNs
!git clone https://github.com/tanushappapogu-max/BINNs.git /content/BINNs
%cd /content/BINNs
!pip install -e '.[ml,imaging]' -q

In [ ]:
# Verify GPU availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Configure paths
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/STS_Project_Data/data')
NIFTI_ROOT = DRIVE_DATA / 'raw' / 'mu_glioma_post'
DERIVED = DRIVE_DATA / 'derived' / 'mu_glioma_post'
MANIFEST = DERIVED / 'shared_split_manifest.json'

ROLE = 'model_selection'  # 'model_selection' for validation (39), 'training' for full (208)
TRANSITION_INDEX = DERIVED / f'shared_{ROLE}_transitions.json'
OUTPUT_ROOT = DRIVE_DATA / f'outputs/pinn_multiscan_{ROLE}'

for p in [NIFTI_ROOT, MANIFEST, TRANSITION_INDEX]:
    assert p.exists(), f'Missing: {p}'
print('All paths verified')

In [ ]:
import json
from gbm_pinn.pinn_cohort import PINNCohortConfig, run_pinn_cohort

config = PINNCohortConfig(
    transition_index_path=TRANSITION_INDEX,
    manifest_path=MANIFEST,
    nifti_root=NIFTI_ROOT,
    output_root=OUTPUT_ROOT,
    data_root=Path('/content/drive/MyDrive/STS_Project_Data'),
    role=ROLE,
    device='cuda',
    resume=True,
)

summary = run_pinn_cohort(config)
print(json.dumps(summary, indent=2))

In [ ]:
# Analyze results
import json
summary = json.load(open(OUTPUT_ROOT / 'cohort_summary.json'))
print(f"Mean Dice: {summary['mean_dice']:.4f}")
print(f"Mean Skill: {summary['mean_dice_skill_over_persistence']:+.4f}")
print(f"Beating Persistence: {summary['n_beating_persistence']}/{summary['n_successful']}")